# 02 - 下载 SEC 财报

这一节只完成一件事：输入股票代码，从 SEC EDGAR 找到并下载最新的 `10-K` 或 `10-Q`。

本节暂时不调用 DeepSeek。先把真实数据获取流程跑通，后面再把它封装成 Agent 工具。

## 1. 配置 SEC 请求身份

SEC 要求请求中包含能够联系到开发者的 `User-Agent`。请在项目的 `.env` 文件中添加一行：

```text
SEC_USER_AGENT=你的名字 your_email@example.com
```

修改 `.env` 后，请重启 Notebook 内核，或者使用 `load_dotenv(override=True)` 重新加载。

In [ ]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(override=True)

SEC_USER_AGENT = os.getenv("SEC_USER_AGENT")
if not SEC_USER_AGENT:
    raise ValueError("请先在 .env 中配置 SEC_USER_AGENT=你的名字 你的邮箱")

SEC_HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
}
DATA_DIR = Path("data/sec")

print("SEC 请求配置完成")

## 2. 创建请求函数

这个函数统一添加请求头、设置超时并检查 HTTP 错误。后续所有 SEC 请求都通过它发送。

In [ ]:
def sec_get(url: str) -> requests.Response:
    response = requests.get(url, headers=SEC_HEADERS, timeout=30)
    response.raise_for_status()
    return response

## 3. 把股票代码转换成 CIK

股票代码例如 `AAPL`，而 SEC 内部使用 CIK 标识公司。这个函数读取 SEC 的公司代码表并完成转换。

In [ ]:
def get_company(ticker: str) -> dict:
    ticker = ticker.strip().upper()
    companies = sec_get(
        "https://www.sec.gov/files/company_tickers.json"
    ).json()

    for company in companies.values():
        if company["ticker"].upper() == ticker:
            return {
                "ticker": ticker,
                "name": company["title"],
                "cik": str(company["cik_str"]).zfill(10),
            }

    raise ValueError(f"SEC 公司列表中没有找到股票代码：{ticker}")

In [ ]:
company = get_company("AAPL")
company

## 4. 查找最新财报

`10-K` 是年度报告，`10-Q` 是季度报告。函数会在公司的近期申报记录中查找指定类型的最新一份。

In [ ]:
def get_latest_filing(ticker: str, form_type: str = "10-K") -> dict:
    company = get_company(ticker)
    submissions_url = (
        f"https://data.sec.gov/submissions/CIK{company['cik']}.json"
    )
    recent = sec_get(submissions_url).json()["filings"]["recent"]

    for index, form in enumerate(recent["form"]):
        if form == form_type:
            accession_number = recent["accessionNumber"][index]
            primary_document = recent["primaryDocument"][index]
            cik_without_zeros = str(int(company["cik"]))
            accession_without_dashes = accession_number.replace("-", "")
            filing_url = (
                "https://www.sec.gov/Archives/edgar/data/"
                f"{cik_without_zeros}/{accession_without_dashes}/{primary_document}"
            )

            return {
                **company,
                "form": form_type,
                "filing_date": recent["filingDate"][index],
                "report_date": recent["reportDate"][index],
                "accession_number": accession_number,
                "primary_document": primary_document,
                "url": filing_url,
            }

    raise ValueError(f"没有在近期申报中找到 {ticker} 的 {form_type}")

In [ ]:
# 可以把 AAPL 换成其他美股代码，把 10-K 换成 10-Q。
TICKER = "AAPL"
FORM_TYPE = "10-K"

filing = get_latest_filing(TICKER, FORM_TYPE)
filing

## 5. 下载财报 HTML

财报会保存到 `data/sec/股票代码/`。保留原始 HTML，后续可以继续做正文提取、分块和引用定位。

In [ ]:
def download_filing(filing: dict) -> Path:
    company_dir = DATA_DIR / filing["ticker"]
    company_dir.mkdir(parents=True, exist_ok=True)

    filename = (
        f"{filing['filing_date']}_{filing['form']}_"
        f"{filing['accession_number']}_{filing['primary_document']}"
    )
    output_path = company_dir / filename

    if output_path.exists():
        print(f"文件已存在，跳过下载：{output_path}")
        return output_path

    response = sec_get(filing["url"])
    output_path.write_bytes(response.content)
    print(f"下载完成：{output_path}")
    return output_path

In [ ]:
filing_path = download_filing(filing)

print(f"公司：{filing['name']}")
print(f"报告类型：{filing['form']}")
print(f"报告期：{filing['report_date']}")
print(f"SEC 原文：{filing['url']}")
print(f"本地文件：{filing_path.resolve()}")

## 这一节完成了什么

现在我们已经有了三个可以继续封装成 Agent 工具的能力：

1. `get_company()`：根据股票代码查找公司和 CIK。
2. `get_latest_filing()`：查找最新的指定类型财报。
3. `download_filing()`：下载原始财报并进行本地缓存。

下一步是解析 HTML，提取可供 DeepSeek 阅读的正文。